# LLM-Powered Semantic Chunking for Advanced Retrieval Augmented Generation (RAG)

In traditional RAG pipelines, text is often split using simple methods like fixed character counts or basic recursive splitting. While functional, these methods frequently fail to respect natural topic boundaries, leading to "context fragmentation"—where a single chunk contains disparate ideas or cuts off critical information mid-sentence. This poor chunking quality severely degrades the performance of the retrieval step, resulting in irrelevant context being passed to the LLM and ultimately diminishing the answer's accuracy.

This notebook introduces an advanced technique: **LLM-based semantic chunking**. Instead of relying on arbitrary delimiters or fixed sizes, we leverage the generative power of a Large Language Model (LLM) to intelligently identify natural topic boundaries within a document. By defining structured output using Pydantic models, we force the LLM not only to split the text but also to generate rich metadata—such as a concise summary for each chunk. This process transforms raw text into highly contextualized, self-describing knowledge units.

Mastering semantic chunking is critical for building robust and production-grade RAG systems. In an advanced workflow managed by LangGraph, this specialized chunking step can be integrated early in the graph's state machine, ensuring that every piece of retrieved context is maximally informative. By learning to structure LLM outputs for data preparation tasks like this, you gain a powerful toolset for moving beyond basic text processing and building truly intelligent knowledge retrieval systems.

### Learning Objectives
Upon completing this notebook, you will be able to:

*   **Implement Structured Output:** Use Pydantic models in conjunction with LangChain's structured output capabilities (`with_structured_output`) to reliably guide an LLM to produce predictable, machine-readable data formats.
*   **Perform Semantic Chunking:** Develop a robust method for splitting large documents into contextually coherent chunks that respect natural topic boundaries, rather than relying on fixed character limits.
*   **Enhance Metadata Generation:** Integrate metadata generation (e.g., summaries) directly into the chunking process, enriching the resulting `Document` objects and improving retrieval relevance.
*   **Prepare for Advanced RAG:** Understand how to transform raw text into high-quality, structured knowledge units suitable for advanced indexing and retrieval within complex LangGraph workflows.


### Setup and Imports

This cell imports necessary libraries for interacting with OpenAI models, defining data structures (Pydantic), handling document objects, loading environment variables, and constructing prompt templates. These tools are foundational for building the LLM components of an advanced RAG system.


In [1]:
from langchain_openai.chat_models import ChatOpenAI # Imports the chat model interface for OpenAI
from pydantic import BaseModel # Used for defining structured output schemas (data validation)
from langchain_core.documents import Document # Represents a chunk of text/document in LangChain
from dotenv import load_dotenv # Utility to load environment variables from a .env file
from langchain_core.prompts import ChatPromptTemplate # Tool for creating reusable and structured chat prompts


In [2]:
load_dotenv()

True

### Code Explanation

This cell initializes a multi-topic string variable (`text`) containing disparate pieces of information (AI/ML, Italian cuisine, and climate change). This diverse content is crucial for demonstrating the robustness of advanced RAG techniques, as it forces the subsequent chunking or retrieval process to handle varied domains and topics.


In [3]:
text = """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future."""

### Structured Output Definition (Pydantic)

This cell defines two Pydantic models (`Chunk` and `Chunker`) to enforce a specific, structured output format for the LLM. This is crucial when you need the model's response to be predictable JSON data rather than free-form text, ensuring that subsequent code can reliably parse the extracted information.


In [4]:
# pydantic class for structured output

# Defines the structure for a single chunk of processed text.
class Chunk(BaseModel): 
    
    chunk_text: str  # The raw or processed text segment.
    summary: str     # A concise summary generated by the LLM.
    
    
# Defines the container class that holds a list of these structured chunks.
class Chunker(BaseModel):
    
    chunks: list[Chunk] # A list containing multiple Chunk objects.


### Model Initialization and Structured Output

This cell initializes the OpenAI chat model (`ChatOpenAI`) and then wraps it using `with_structured_output`. This is crucial because it forces the LLM to generate its output not as free text, but according to a predefined Pydantic schema (`Chunker`), ensuring reliable data parsing for subsequent steps.


In [5]:
# define model

# Initialize the ChatOpenAI model instance.
model = ChatOpenAI(model="gpt-5-mini")

# Use with_structured_output to force the LLM's output into a specific format (defined by Chunker).
llm_chunker = model.with_structured_output(schema=Chunker)


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

### Prompt Definition for Advanced Chunking

This cell defines a structured prompt using `ChatPromptTemplate`. It instructs an LLM to act as an expert text chunker, ensuring that the output is a list of strings (chunks), maintaining the original text integrity, and crucially, generating a 1-2 line summary for each resulting chunk. This approach leverages the LLM's understanding of natural topic boundaries for superior document splitting compared to simple fixed-size methods.


In [ ]:
# prompt for chunking

prompt = ChatPromptTemplate(messages=[
    ("system", 
     """You are an expert Text Chunker that splits the given text and outputs them as a 
     list of strings. You understand the natural topic boundaries of text and 
     also do not change the existing text. You just split the text where ever applicable.
     Once you create the chunk, you also generate a 1-2 line summary of the chunk also"""),
    ("human",
     "Split the given text into chunks\nText: {text}")
], input_variables=["text"])

### Chunking through LLM

This cell utilizes a defined `model_chain` (likely an LCEL chain) to perform advanced text chunking. By passing the input `text` through this chain, it leverages the LLM's understanding to intelligently segment the document content, which is superior to simple fixed-size splitting.


In [ ]:
# chunking through llm

# Define and invoke the model chain for intelligent chunking.
model_chain = prompt | llm_chunker

# Invoke the chain with the input text to get the intelligently chunked response.
response = model_chain.invoke({"text": text})


This cell is used to display the output of the preceding code block, which typically contains a structured object (like a list, dictionary, or DataFrame) that needs visualization. It confirms successful execution and allows the user to inspect the generated data structure.


In [ ]:
response
# This line simply displays the value stored in the 'response' variable.
# In a Jupyter notebook context, this is used to visualize the output of the preceding computation,
# confirming that the desired data (e.g., split chunks, embeddings) was successfully generated.


Chunker(chunks=[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure.", summary="Overview of artificial intelligence topics: AI's impact, machine learning, deep learning, and neural networks."), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.', summary='Notes on pasta and Italian cuisine: ingredients, authentic carbonara, and cooking pasta al dente.'), Chunk(chunk_text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\n

### Code Explanation

This cell accesses the `chunks` attribute of the `response` object. Assuming `response` is an object returned by a document loading or processing function (like those from LangChain), this attribute contains a list or iterable of text chunks, which are smaller, manageable pieces of the original document used for embedding and retrieval.


In [ ]:
response.chunks


[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure.", summary="Overview of artificial intelligence topics: AI's impact, machine learning, deep learning, and neural networks."),
 Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.', summary='Notes on pasta and Italian cuisine: ingredients, authentic carbonara, and cooking pasta al dente.'),
 Chunk(chunk_text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\nScientists wa

This cell calculates and displays the total number of chunks (or segments) contained within the `response` object. This is crucial for verifying that the chunking process successfully segmented the original document into the expected number of pieces, which informs subsequent retrieval steps.


In [ ]:
len(response.chunks) # Calculates the length of the 'chunks' attribute (a list or sequence) within the 'response' object to count the total number of generated chunks.


3

### Code Explanation

This line accesses the list of text chunks generated by the `response` object. The `response` object, typically returned from a document loading or splitting process (like those involving LangChain's TextSplitters), encapsulates the original content and its subsequent divisions into smaller, manageable pieces for embedding and retrieval.


In [ ]:
chunks = response.chunks # Accesses the list of text chunks stored within the 'response' object.


In [ ]:
from termcolor import COLORS, colored
from random import choice

### Chunk Visualization and Inspection

This function iterates through a list of text chunks, printing the total count and then displaying each chunk's length and content. It uses color-coding (via `colored` and predefined `COLORS`) to visually distinguish the chunks, which is crucial for debugging and inspecting the output quality of the splitting process.


In [ ]:
def display_chunks(chunks):
    # Select a subset of colors from the global COLORS dictionary for visualization
    colors_list = list(COLORS.keys())[2:8]
    # Print the total number of chunks processed
    print(f"Total Number of Chunks: {len(chunks)}")
    
    # Iterate through the chunks, using enumerate to get both index (num) and chunk content
    for num, chunk in enumerate(chunks, 1):
        # Print the chunk number and its character length
        print(f"Chunk {num}: Length {len(chunk)} chars")
        # Print the actual chunk text, applying a random color from colors_list for visual separation
        print(colored(text=chunk, color=choice(colors_list)), end="\n\n")


### Code Explanation

This cell uses a list comprehension to extract the raw text content (`chunk_text`) from every `Chunk` object within the `chunks` list. The resulting list of strings is then passed to the `display_chunks` function, which visualizes or prints these individual chunks for inspection.


In [ ]:
display_chunks([chunk.chunk_text for chunk in chunks])


Total Number of Chunks: 3
Chunk 1: Length 277 chars
Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.

Chunk 2: Length 283 chars
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.

Chunk 3: Length 260 chars
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future.



### Code Explanation

This cell accesses and displays the third chunk (index 2) from the `chunks` list attribute of the `response` object. This is typically used for inspecting the structure or content of the chunks generated by a document splitter, allowing the user to verify that the splitting process worked as expected.


In [ ]:
response.chunks[2]


Chunk(chunk_text='Climate change is affecting ecosystems worldwide.\nRising temperatures are causing glaciers to melt at unprecedented rates.\nScientists warn that immediate action is needed to reduce carbon emissions.\nRenewable energy sources offer hope for a sustainable future.', summary='Discussion of climate change impacts, the urgency of emissions reduction, and the role of renewable energy.')

### Document Conversion

This cell converts a list of raw text chunks (presumably generated by a splitter) into a list of `Document` objects. This standardized format is crucial for subsequent RAG steps, as it packages the chunk content (`page_content`) along with relevant metadata (like a summary).

* **Key Class:** `Document` (from LangChain/LlamaIndex context).


In [ ]:
# create documents from chunks

# We iterate through the list of raw text chunks and convert each one into a Document object.
# The chunk's text is assigned to 'page_content', and its summary is stored in the metadata dictionary.
docs = [Document(page_content=chunk.chunk_text, metadata={"summary": chunk.summary}) for chunk in chunks]


In [ ]:
print(docs)

[Document(metadata={'summary': "Overview of artificial intelligence topics: AI's impact, machine learning, deep learning, and neural networks."}, page_content="Artificial intelligence is transforming technology and shaping the future.\nMachine learning algorithms are becoming more sophisticated every day.\nDeep learning models can now process vast amounts of data efficiently.\nNeural networks are inspired by the human brain's structure."), Document(metadata={'summary': 'Notes on pasta and Italian cuisine: ingredients, authentic carbonara, and cooking pasta al dente.'}, page_content='The best pasta recipes include fresh ingredients and proper cooking techniques.\nItalian cuisine emphasizes quality olive oil and regional cheeses.\nAuthentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.\nCooking pasta al dente ensures the best texture and flavor.'), Document(metadata={'summary': 'Discussion of climate change impacts, the urgency of emissions reduction, and the role of

In [ ]:
print(docs[1])

page_content='The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.' metadata={'summary': 'Notes on pasta and Italian cuisine: ingredients, authentic carbonara, and cooking pasta al dente.'}
